In [15]:


# Bike Sharing Demand Prediction - Assignment Notebook
# Filename: bike_sharing_assignment.py
# This script/notebook performs EDA, feature engineering, modeling and generates submission.csv
# Place bike_train.csv and bike_test.csv in the same folder as this script and run it.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# ----------------------- Utility functions -----------------------

def rmsle(y_true, y_pred):
    # ensure non-negative
    y_pred = np.maximum(0, y_pred)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

# ----------------------- Load data -----------------------
train_path = "C:\\Users\\Santhoshn\\Downloads\\bike_train.csv"
test_path = "C:\\Users\\Santhoshn\\Downloads\\bike_test.csv"


if not os.path.exists(train_path) or not os.path.exists(test_path):
    raise FileNotFoundError('Please place bike_train.csv and bike_test.csv in the script folder before running.')

train['datetime'] = pd.to_datetime(train['datetime'], errors='coerce')
test['datetime'] = pd.to_datetime(test['datetime'], errors='coerce')

# Optional sampleSubmission handling (if present in folder)
sample_sub_path = 'sampleSubmission.csv'
if os.path.exists(sample_sub_path):
    sample_sub = pd.read_csv(sample_sub_path)
else:
    sample_sub = None

# ----------------------- Q1. Dataset size, missing values, feature types -----------------------
print('\nQ1. Dataset overview')
print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('\nTrain info:')
print(train.info())
print('\nMissing values in train:\n', train.isnull().sum())
print('\nData types:\n', train.dtypes)

# ----------------------- Q2. Visualizations: relationships with target -----------------------
# We'll create a few simple plots and save them to files.
plots_dir = 'plots'
os.makedirs(plots_dir, exist_ok=True)

# Extract time features for plotting
for df in [train, test]:
    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.weekday
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year

# Plot mean count by hour
plt.figure(figsize=(8,4))
train.groupby('hour')['count'].mean().plot(marker='o')
plt.title('Average bike rentals by hour')
plt.xlabel('Hour of day')
plt.ylabel('Average count')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'avg_count_by_hour.png'))
plt.close()

# Plot mean count by weekday
plt.figure(figsize=(8,4))
train.groupby('weekday')['count'].mean().plot(marker='o')
plt.title('Average bike rentals by weekday (0=Mon)')
plt.xlabel('Weekday')
plt.ylabel('Average count')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'avg_count_by_weekday.png'))
plt.close()

# Scatter of temp vs count
plt.figure(figsize=(6,4))
plt.scatter(train['temp'], train['count'], alpha=0.3, s=8)
plt.title('Temp vs Count')
plt.xlabel('Temperature')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'temp_vs_count.png'))
plt.close()

print('\nQ2. Plots saved to ./plots (avg_count_by_hour.png, avg_count_by_weekday.png, temp_vs_count.png)')

# ----------------------- Q3. Informative variables (initial guess) -----------------------
print('\nQ3. Likely informative variables:')
print("- hour (time of day), weekday, month: strong time-of-day and weekly patterns")
print("- weather features: temp, humidity, windspeed")
print("- season (if provided) and holiday/workingday indicators")

# ----------------------- Q4. Feature engineering -----------------------
# We'll create common features: hour, is_peak (morning/evening), sin/cos transforms for hour to capture cyclical nature

def add_features(df):
    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.weekday
    df['month'] = df['datetime'].dt.month
    df['day'] = df['datetime'].dt.day
    df['year'] = df['datetime'].dt.year
    # cyclical encoding for hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    # peak indicator
    df['is_morning_peak'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_evening_peak'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)
    return df

train = add_features(train)
test = add_features(test)

print('\nQ4. Feature engineering performed: hour, weekday, month, cyclical hour sin/cos, peak flags')

# ----------------------- Prepare data for modeling -----------------------
# We'll separate features and target. For baseline, use a mix of numeric and categorical features.

target = 'count'
features = ['hour', 'weekday', 'month', 'temp', 'humidity', 'windspeed', 'season', 'holiday', 'workingday',
            'hour_sin', 'hour_cos', 'is_morning_peak', 'is_evening_peak']

# If 'season', 'holiday', 'workingday' are integers, treat them as categorical

X = train[features].copy()
y = train[target].copy()
X_test = test[features].copy()

# Split train/validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ----------------------- Preprocessing pipeline -----------------------
num_features = ['temp', 'humidity', 'windspeed', 'hour_sin', 'hour_cos']
cat_features = ['hour', 'weekday', 'month', 'season', 'holiday', 'workingday', 'is_morning_peak', 'is_evening_peak']

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
cat_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# ----------------------- Q5. Baseline Linear Regression -----------------------
baseline_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])

print('\nTraining baseline Linear Regression...')
baseline_pipe.fit(X_train, y_train)

y_pred_val = baseline_pipe.predict(X_val)
print('Baseline RMSLE on validation:', rmsle(y_val, y_pred_val))

# ----------------------- Q6. Polynomial features + Ridge and Lasso -----------------------
# We'll try PolynomialFeatures on numeric features to degree=2 and then ridge/lasso with grid search

# Build a pipeline that expands numeric features using polynomial basis (degree 2) then standardize and fit ridge/lasso
poly_num = Pipeline(steps=[('poly', PolynomialFeatures(degree=2, include_bias=False)), ('scaler', StandardScaler())])
poly_preprocessor = ColumnTransformer(transformers=[
    ('poly_num', poly_num, num_features),
    ('cat', cat_transformer, cat_features)
])

# Ridge
ridge_pipe = Pipeline(steps=[('pre', poly_preprocessor), ('ridge', Ridge())])
params_ridge = {'ridge__alpha': [0.1, 1.0, 10.0, 50.0]}

print('\nGrid searching Ridge over alpha...')
ridge_search = GridSearchCV(ridge_pipe, params_ridge, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_search.fit(X_train, y_train)

best_ridge = ridge_search.best_estimator_
print('Best ridge alpha:', ridge_search.best_params_)
print('Ridge RMSLE on validation:', rmsle(y_val, best_ridge.predict(X_val)))

# Lasso
lasso_pipe = Pipeline(steps=[('pre', poly_preprocessor), ('lasso', Lasso(max_iter=5000))])
params_lasso = {'lasso__alpha': [0.001, 0.01, 0.1, 1.0]}

print('\nGrid searching Lasso over alpha...')
lasso_search = GridSearchCV(lasso_pipe, params_lasso, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
lasso_search.fit(X_train, y_train)

best_lasso = lasso_search.best_estimator_
print('Best lasso alpha:', lasso_search.best_params_)
print('Lasso RMSLE on validation:', rmsle(y_val, best_lasso.predict(X_val)))

# ----------------------- Q7. Summarize results -----------------------
results = []
results.append(('LinearRegression', rmsle(y_val, y_pred_val)))
results.append(('Ridge (poly d2)', rmsle(y_val, best_ridge.predict(X_val))))
results.append(('Lasso (poly d2)', rmsle(y_val, best_lasso.predict(X_val))))

results_df = pd.DataFrame(results, columns=['Model', 'RMSLE_val'])
print('\nQ7. Model comparison:')
print(results_df)
results_df.to_csv('model_comparison.csv', index=False)

# ----------------------- Q8. Residuals for best model -----------------------
# Choose best model by RMSLE
best_model_name = results_df.loc[results_df['RMSLE_val'].idxmin(), 'Model']
if best_model_name == 'LinearRegression':
    chosen_model = baseline_pipe
elif best_model_name == 'Ridge (poly d2)':
    chosen_model = best_ridge
else:
    chosen_model = best_lasso

residuals = y_val - chosen_model.predict(X_val)
plt.figure(figsize=(8,5))
plt.scatter(chosen_model.predict(X_val), residuals, alpha=0.4)
plt.hlines(0, xmin=chosen_model.predict(X_val).min(), xmax=chosen_model.predict(X_val).max(), colors='r')
plt.xlabel('Predicted count')
plt.ylabel('Residual (actual - predicted)')
plt.title(f'Residual plot for {best_model_name}')
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'residual_plot_best_model.png'))
plt.close()

print('\nQ8. Residual plot saved to ./plots/residual_plot_best_model.png')

# ----------------------- Q9. Explanation (printed in console) -----------------------
print('\nQ9. Explanation why the winning model performs better:')
print('- If polynomial + regularized model (Ridge/Lasso) wins, it likely captured non-linear effects (temp^2, interaction terms)')
print('- Regularization reduces overfitting from expanded polynomial feature set')
print('- Cyclical hour encoding (sin/cos) helps capture time-of-day patterns better than raw hour alone')

# ----------------------- Q10-Q12 Reflection answers -----------------------
print('\nQ10-Q12: Reflection answers (brief):')
print('\nQ10. Why RMSLE penalizes under-predictions more gently than RMSE?')
print('RMSLE uses log(pred+1) and log(actual+1). The log transform compresses large values, so differences at high counts are reduced relative to RMSE; under-predictions and over-predictions are measured on a multiplicative/log scale, so RMSLE is less sensitive to absolute differences when actual values are large.')

print('\nQ11. Trade-offs between simplicity and predictive power:')
print('- Simple models (linear) are interpretable, faster, and less likely to overfit, but may miss non-linear patterns')
print('- Complex models (polynomial, ensembles) capture more structure but need regularization, more tuning, and are less interpretable')

print('\nQ12. Why Linear Regression alone may not capture time-of-day effects:')
print("- Time-of-day is cyclical: hour 23 and hour 0 are adjacent but linear encoding treats them as far apart. Linear regression on raw hour can't model cyclicity without transformations (sin/cos or one-hot). Also interactions between hour and other variables (weather) are non-linear.")

# ----------------------- Q11. Create submission on test set using chosen model -----------------------
print('\nCreating submission using the best model...')

# Fit the chosen model on full training data (train) before predicting test
X_full = train[features]
y_full = train[target]

if best_model_name == 'LinearRegression':
    final_model = baseline_pipe
    final_model.fit(X_full, y_full)
elif best_model_name == 'Ridge (poly d2)':
    final_model = ridge_search.best_estimator_
    final_model.fit(X_full, y_full)
else:
    final_model = lasso_search.best_estimator_
    final_model.fit(X_full, y_full)

# Predict on test set
preds_test = final_model.predict(X_test)
# Ensure no negative predictions
preds_test = np.maximum(0, preds_test)

# Prepare submission file (use sampleSubmission if available otherwise use datetime as id)
if sample_sub is not None and 'count' in sample_sub.columns:
    submission = sample_sub.copy()
    submission['count'] = preds_test
else:
    submission = pd.DataFrame({'datetime': test['datetime'], 'count': preds_test})

submission.to_csv('submission.csv', index=False)
print('Submission saved to submission.csv')

print('\nAll done. You will find:')
print('- model_comparison.csv (summary of RMSLE)')
print('- submission.csv (predictions on test)')
print("- plots/ (visualizations and residual plot)")

# End of script



Q1. Dataset overview
Train shape: (10450, 16)
Test shape: (2613, 9)

Train info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10450 entries, 0 to 10449
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    10450 non-null  datetime64[ns]
 1   season      10450 non-null  int64         
 2   holiday     10450 non-null  int64         
 3   workingday  10450 non-null  int64         
 4   weather     10450 non-null  int64         
 5   temp        10450 non-null  float64       
 6   atemp       10450 non-null  float64       
 7   humidity    10450 non-null  int64         
 8   windspeed   10450 non-null  float64       
 9   casual      10450 non-null  int64         
 10  registered  10450 non-null  int64         
 11  count       10450 non-null  int64         
 12  hour        10450 non-null  int32         
 13  weekday     10450 non-null  int32         
 14  month       10450 non-null  int32   

C:\Program Files\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.427e+07, tolerance: 2.739e+04
  model = cd_fast.enet_coordinate_descent(


Best lasso alpha: {'lasso__alpha': 0.001}
Lasso RMSLE on validation: 1.0136285115037718

Q7. Model comparison:
              Model  RMSLE_val
0  LinearRegression   1.047389
1   Ridge (poly d2)   1.011508
2   Lasso (poly d2)   1.013629

Q8. Residual plot saved to ./plots/residual_plot_best_model.png

Q9. Explanation why the winning model performs better:
- If polynomial + regularized model (Ridge/Lasso) wins, it likely captured non-linear effects (temp^2, interaction terms)
- Regularization reduces overfitting from expanded polynomial feature set
- Cyclical hour encoding (sin/cos) helps capture time-of-day patterns better than raw hour alone

Q10-Q12: Reflection answers (brief):

Q10. Why RMSLE penalizes under-predictions more gently than RMSE?
RMSLE uses log(pred+1) and log(actual+1). The log transform compresses large values, so differences at high counts are reduced relative to RMSE; under-predictions and over-predictions are measured on a multiplicative/log scale, so RMSLE is less s

ValueError: Input X contains NaN.
PolynomialFeatures does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values